# LLM Roundtable — human framing

**Critique** and **improvement** prompts frame peer models as **humans**, while primary and merger stay shared.

**n** models answer a prompt → each response is critiqued by the other **n−1** → each author refines using each critique → refinements are **merged** per author → compare **merged** vs **primary** (TF-IDF cosine).

Edit the **Configuration** cell below (models, API key, and all phase prompts), then run the setup cells underneath before running the pipeline.


In [ ]:
# =============================================================================
# Configuration — edit this cell only
# =============================================================================
import os

# API key: set here or export OPENROUTER_API_KEY in your environment
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
# OPENROUTER_API_KEY = "sk-or-..."

# OpenRouter model slugs (one per roundtable participant)
MODELS = [
    "openai/gpt-5.4",
    "anthropic/claude-opus-4.6",
    "google/gemini-2.5-pro",
    "deepseek/deepseek-r1",
]

# Model that merges each author's n−1 refinements into one answer
MERGER_MODEL = "openai/gpt-5.4"

# Sampling and parallelism
TEMPERATURE = 0.5
MAX_WORKERS = 8
MAX_TOKENS = 8192

# -----------------------------------------------------------------------------
# Prompts — edit any of these (placeholders documented inline)
# -----------------------------------------------------------------------------

# Phase 1: user message sent to each author model
PRIMARY_PROMPT = """You are an expert in clinical terminology and medical knowledge representation.

Design a formula that measures how useful a candidate concept would be in an existing target ontology. The score will determine whether or not to insert the concept into the ontology. Identify and explain the factors the formula should consider.

I am building a framework that decides whether a new concept should be added to an existing ontology. A high-utility concept should be inserted, whereas a low-utility concept should not.

Treat utility as a function of the current content of the target ontology only, such as Medical Subject Headings (MeSH), Systematized Nomenclature of Medicine - Clinical Terms (SNOMED CT), Medical Dictionary for Regulatory Activities Terminology (MedDRA), International Classification of Diseases - 10th Revision - Clinical Modification (ICD10-CM), National Cancer Institute Thesaurus (NCIt), Online Mendelian Inheritance in Man (OMIM), Biological and Environmental Research Ontology (BERO) and Gene Ontology (GO).

Provide a justification for every factor in the formula. For each factor, explain:
What does it measure?
Why is it important for evaluating the utility of a candidate concept?
How does it influence the final score?
Have any assumptions been made about its role in the formula?
Briefly note any factors you excluded and why.

Return your answer in the following format:
    1. The proposed formula in plain text.
    2. A list of the factors it considers.
    3. Any assumptions or limitations of the formula.
"""

# Phase 2: critique prompt (HUMAN framing)
# Placeholders: {prompt}, {response_to_evaluate}
CRITIQUE_TEMPLATE = """You are an experienced critic in the domain of biomedical ontologies evaluating someone else's answer.

Critique the answer below: What are its strengths, weaknesses, and areas for improvement? For each weakness, explain how it can be improved.

I posed the following problem to another human expert:
	ORIGINAL PROMPT: {prompt}
They returned this result:
ANSWER: {response_to_evaluate}

Be concise and specific with every point, avoiding vague praise or criticism. Address each flaw by stating what is wrong and how to improve on it.

Return your answer in the following format:
1. Strengths
2. Weaknesses and gaps
3. Concrete improvement recommendations (how, not just what)
"""

# Phase 3: refinement / improvement prompt (HUMAN framing)
# Placeholders: {prompt}, {original_response}, {critique}
REFINEMENT_TEMPLATE = """You are the original author revising your own work in response to expert feedback.

Produce an improved version of your original answer that incorporates the critique below.

I previously asked you:
ORIGINAL PROMPT: {prompt}
You responded with the following:
YOUR ANSWER: {original_response}
I passed your response to a human for critique. They returned:
CRITIQUE: {critique}

Address each point raised in the critique; if you disagree with a point, say so and justify why. Keep what was already strong; do not regress on correct material.

Maintain the original output requirements (plain-text notation, formula + factors + assumptions).
"""

# Phase 4: how each revised version is formatted inside the merge prompt
# Placeholders: {version_number}, {text}
# Blind label (1-based critic slot). Never include model slugs or "LLM".
MERGE_VERSION_BLOCK_TEMPLATE = """### Version {version_number}
{text}
"""

# Phase 4: merge prompt
# Placeholders: {prompt}, {revised_versions}
MERGE_TEMPLATE = """You are a response aggregator. Several revised answers to the same prompt were produced by one author after it incorporated several independent critiques. Your job is to merge them into one definitive answer.

Merge the revised versions below into ONE final answer that is at least as good as any individual version in every aspect. Each version was the same author's attempt to improve the same underlying answer in response to a different critique.

The author was originally asked:
ORIGINAL PROMPT: {prompt}
The author then produced the following revised versions, each one incorporating a different critique:
REVISED VERSIONS: {revised_versions}

Unify notation, terminology, and structure so the result reads as one consistently authored response rather than stitched-together drafts. Where revisions conflict, keep the better-justified claim and drop the weaker one. Preserve every correct improvement that any revision introduced; do not regress to a weaker formulation that an earlier revision had already fixed. Judge content on its merits, not on which critique prompted it. You are not bound to any single version as a base. Do not mention the merge, the existence of multiple drafts, or the critiques. Write as if this were a single original answer to the prompt.

Output a single coherent answer. Do not output multiple versions, a difference report, or a side-by-side comparison. Maintain the original output requirements (plain-text notation, formula + factors + assumptions).
"""

# Output and display options
OUTPUT_DIR = "outputs/human"
AUTO_SAVE = True
SHOW_INTERMEDIATES = True  # show critiques & refinements in the final cell


## Imports and helpers

Run this cell once after editing configuration. No edits needed here.


In [ ]:
from __future__ import annotations

import concurrent.futures
import json
import math
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from openai import OpenAI
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
TFIDF_MAX_FEATURES = 4096
TFIDF_STOP_WORDS = "english"

CRITIQUE_TEMPLATE_PLACEHOLDERS = (
    "{prompt}",
    "{response_to_evaluate}",
)

REFINEMENT_TEMPLATE_PLACEHOLDERS = (
    "{prompt}",
    "{original_response}",
    "{critique}",
)

MERGE_VERSION_BLOCK_PLACEHOLDERS = (
    "{version_number}",
    "{text}",
)

MERGE_TEMPLATE_PLACEHOLDERS = (
    "{prompt}",
    "{revised_versions}",
)

def get_client(api_key: str) -> OpenAI:
    return OpenAI(
        base_url=OPENROUTER_BASE,
        api_key=api_key,
        default_headers={
            "HTTP-Referer": "https://github.com/narenkhatwani/llm-roundtable",
            "X-Title": "LLM Roundtable",
        },
    )


def chat_complete(
    client: OpenAI,
    model: str,
    messages: list[dict[str, str]],
    temperature: float,
    max_tokens: int = MAX_TOKENS,
) -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    content = resp.choices[0].message.content
    return (content or "").strip()


def _format_template(template: str, placeholders: tuple[str, ...], **kwargs: str) -> str:
    try:
        return template.format(**kwargs)
    except KeyError as e:
        raise ValueError(
            f"Template contains unknown placeholder {e}. "
            f"Use only: {', '.join(placeholders)}"
        ) from e


def critique_user_message(
    template: str,
    original_prompt: str,
    responder_label: str,
    response_text: str,
) -> str:
    # responder_label kept for call-site compatibility; critique prompt omits it
    _ = responder_label
    return _format_template(
        template,
        CRITIQUE_TEMPLATE_PLACEHOLDERS,
        prompt=original_prompt,
        response_to_evaluate=response_text,
    )


def refine_user_message(
    template: str,
    original_prompt: str,
    original_response: str,
    critic_label: str,
    critique_text: str,
) -> str:
    # critic_label kept for call-site compatibility; refine prompt omits it
    _ = critic_label
    return _format_template(
        template,
        REFINEMENT_TEMPLATE_PLACEHOLDERS,
        prompt=original_prompt,
        original_response=original_response,
        critique=critique_text,
    )


def merge_user_message(
    merge_template: str,
    version_block_template: str,
    original_prompt: str,
    author_label: str,
    refined_blocks: list[tuple[str, str]],
) -> str:
    # author_label kept for call-site compatibility; merge prompt omits it
    _ = author_label
    blocks = [
        _format_template(
            version_block_template,
            MERGE_VERSION_BLOCK_PLACEHOLDERS,
            version_number=version_number,
            text=text,
        )
        for version_number, text in refined_blocks
    ]
    return _format_template(
        merge_template,
        MERGE_TEMPLATE_PLACEHOLDERS,
        prompt=original_prompt,
        revised_versions="\n".join(blocks),
    )

def _tfidf_vectorizer_kwargs(**overrides: object) -> dict:
    base: dict = {
        "max_features": TFIDF_MAX_FEATURES,
        "stop_words": TFIDF_STOP_WORDS,
        "smooth_idf": True,
        "sublinear_tf": False,
    }
    base.update(overrides)
    return base


def tfidf_cosine(a: str, b: str) -> float:
    if not a.strip() or not b.strip():
        return 0.0
    vec = TfidfVectorizer(**_tfidf_vectorizer_kwargs(norm="l2"))
    try:
        m = vec.fit_transform([a, b])
        return float(cosine_similarity(m[0:1], m[1:2])[0, 0])
    except ValueError:
        return 0.0


def tfidf_pair_breakdown(text_a: str, text_b: str, top_n: int = 18) -> dict | None:
    if not text_a.strip() or not text_b.strip():
        return None
    kw = _tfidf_vectorizer_kwargs()
    try:
        vec_raw = TfidfVectorizer(**kw, norm=None)
        X = vec_raw.fit_transform([text_a, text_b]).toarray()
    except ValueError:
        return None

    idf = np.asarray(vec_raw.idf_, dtype=float)
    names = vec_raw.get_feature_names_out()
    n_samples = 2

    cv = CountVectorizer(vocabulary=vec_raw.vocabulary_)
    C = cv.fit_transform([text_a, text_b])
    df = np.asarray(C.astype(bool).sum(axis=0)).ravel().astype(int)

    idf_from_df = np.log((n_samples + 1) / (df + 1)) + 1.0
    idf_matches = bool(np.allclose(idf, idf_from_df))

    w = X.copy()
    tf = np.zeros_like(w)
    safe = idf > 1e-15
    tf[:, safe] = w[:, safe] / idf[safe]

    n0 = float(np.linalg.norm(w[0]))
    n1 = float(np.linalg.norm(w[1]))
    cosine_manual = float(np.dot(w[0], w[1]) / (n0 * n1)) if n0 > 1e-15 and n1 > 1e-15 else 0.0

    vec_l2 = TfidfVectorizer(**kw, norm="l2")
    Xl2 = vec_l2.fit_transform([text_a, text_b])
    cosine_display = float(cosine_similarity(Xl2[0:1], Xl2[1:2])[0, 0])

    def top_rows(doc_idx: int) -> list[dict]:
        row = w[doc_idx]
        order = np.argsort(-np.abs(row))
        out: list[dict] = []
        for j in order[:top_n]:
            if row[j] == 0.0:
                continue
            tfc = tf[doc_idx, j]
            tfc_i = int(round(tfc)) if abs(tfc - round(tfc)) < 1e-5 else float(tfc)
            out.append(
                {
                    "term": str(names[j]),
                    "tf (count)": tfc_i,
                    "df (of 2 docs)": int(df[j]),
                    "idf": float(idf[j]),
                    "tf × idf": float(row[j]),
                }
            )
        return out

    sample_terms: list[dict] = []
    active = np.where((w[0] != 0) | (w[1] != 0))[0]
    for j in active[:8]:
        sample_terms.append(
            {
                "term": str(names[j]),
                "df": int(df[j]),
                "idf (sklearn)": float(idf[j]),
                "ln((1+n)/(1+df))+1": float(math.log((1 + n_samples) / (1 + df[j])) + 1),
            }
        )

    return {
        "vocab_size": int(len(names)),
        "cosine_l2": cosine_display,
        "cosine_manual_raw_vectors": cosine_manual,
        "cosine_match": bool(abs(cosine_display - cosine_manual) < 1e-5),
        "primary_top": top_rows(0),
        "merged_top": top_rows(1),
        "idf_matches_df_formula": idf_matches,
        "sample_idf_rows": sample_terms[:6],
    }

def build_run_bundle(
    run_id: str,
    models: list[str],
    merger_model: str,
    temperature: float,
    max_workers: int,
    user_prompt: str,
    critique_template: str,
    refinement_template: str,
    merge_template: str,
    merge_version_block_template: str,
    primary: list[str | None],
    critiques: list[list[str | None]],
    refined: list[list[str | None]],
    merged: list[str | None],
) -> dict:
    return {
        "run_id": run_id,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
        "models": models,
        "merger_model": merger_model,
        "temperature": temperature,
        "max_workers": max_workers,
        "primary_prompt": user_prompt,
        "critique_template": critique_template,
        "refinement_template": refinement_template,
        "merge_template": merge_template,
        "merge_version_block_template": merge_version_block_template,
        "primary_responses": primary,
        "critiques": critiques,
        "refined_responses": refined,
        "merged_responses": merged,
    }


def save_run_bundle_to_dir(bundle: dict, base_dir: str) -> Path:
    root = Path(base_dir).expanduser().resolve()
    run_dir = root / bundle["run_id"]
    run_dir.mkdir(parents=True, exist_ok=True)
    out = run_dir / "roundtable_run.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(bundle, f, ensure_ascii=False, indent=2)
    return run_dir


def validate_prompt_templates(
    critique_template: str,
    refinement_template: str,
    merge_template: str,
    merge_version_block_template: str,
) -> None:
    critique_user_message(critique_template, "PROMPT", "RESPONDER", "RESPONSE")
    refine_user_message(refinement_template, "PROMPT", "RESPONSE", "CRITIC", "CRITIQUE")
    _format_template(
        merge_version_block_template,
        MERGE_VERSION_BLOCK_PLACEHOLDERS,
        version_number="1",
        text="TEXT",
    )
    merge_user_message(
        merge_template,
        merge_version_block_template,
        "PROMPT",
        "AUTHOR",
        [("1", "TEXT")],
    )


def estimate_api_calls(n: int) -> int:
    return n + n * (n - 1) + n * (n - 1) + n

## Pipeline runners

Parallel phase functions with tqdm progress bars.


In [ ]:
def run_primary(
    client: OpenAI,
    models: list[str],
    user_prompt: str,
    temperature: float,
    max_workers: int,
) -> list[str | None]:
    n = len(models)
    out: list[str | None] = [None] * n

    def one(i: int) -> tuple[int, str]:
        text = chat_complete(
            client,
            models[i],
            [{"role": "user", "content": user_prompt}],
            temperature,
        )
        return i, text

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(one, i) for i in range(n)]
        for fut in tqdm(
            concurrent.futures.as_completed(futs),
            total=len(futs),
            desc="Phase 1: primary responses",
        ):
            i, text = fut.result()
            out[i] = text
    return out


def run_critiques(
    client: OpenAI,
    models: list[str],
    primary: list[str],
    user_prompt: str,
    critique_template: str,
    temperature: float,
    max_workers: int,
) -> list[list[str | None]]:
    n = len(models)
    grid: list[list[str | None]] = [[None] * n for _ in range(n)]
    tasks = [(i, j) for i in range(n) for j in range(n) if j != i]

    def work(i: int, j: int) -> tuple[int, int, str]:
        label_i = f"author {i + 1}"
        msg = critique_user_message(
            critique_template,
            user_prompt,
            label_i,
            primary[i] or "",
        )
        text = chat_complete(
            client,
            models[j],
            [{"role": "user", "content": msg}],
            temperature,
        )
        return i, j, text

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(work, i, j) for i, j in tasks]
        for fut in tqdm(
            concurrent.futures.as_completed(futs),
            total=len(futs),
            desc="Phase 2: cross-model critiques",
        ):
            i, j, text = fut.result()
            grid[i][j] = text
    return grid


def run_refine(
    client: OpenAI,
    models: list[str],
    primary: list[str],
    critiques: list[list[str | None]],
    user_prompt: str,
    refinement_template: str,
    temperature: float,
    max_workers: int,
) -> list[list[str | None]]:
    n = len(models)
    refined: list[list[str | None]] = [[None] * n for _ in range(n)]
    tasks = [(i, j) for i in range(n) for j in range(n) if j != i]

    def work(i: int, j: int) -> tuple[int, int, str]:
        critic_label = f"critic {j + 1}"
        msg = refine_user_message(
            refinement_template,
            user_prompt,
            primary[i] or "",
            critic_label,
            critiques[i][j] or "",
        )
        text = chat_complete(
            client,
            models[i],
            [{"role": "user", "content": msg}],
            temperature,
        )
        return i, j, text

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(work, i, j) for i, j in tasks]
        for fut in tqdm(
            concurrent.futures.as_completed(futs),
            total=len(futs),
            desc="Phase 3: refinements",
        ):
            i, j, text = fut.result()
            refined[i][j] = text
    return refined


def run_merges(
    client: OpenAI,
    merger_model: str,
    models: list[str],
    refined: list[list[str | None]],
    user_prompt: str,
    merge_template: str,
    merge_version_block_template: str,
    temperature: float,
) -> list[str | None]:
    n = len(models)
    merged: list[str | None] = [None] * n
    for i in tqdm(range(n), desc="Phase 4: merge per author"):
        blocks: list[tuple[str, str]] = []
        for j in range(n):
            if j == i:
                continue
            t = refined[i][j]
            if t:
                blocks.append((str(j + 1), t))
        if not blocks:
            merged[i] = None
            continue
        author_label = f"author {i + 1}"
        msg = merge_user_message(
            merge_template,
            merge_version_block_template,
            user_prompt,
            author_label,
            blocks,
        )
        merged[i] = chat_complete(
            client,
            merger_model,
            [{"role": "user", "content": msg}],
            temperature,
        )
    return merged


print("Helpers loaded.")
print(f"Models: {len(MODELS)} | Merger: {MERGER_MODEL} | Workers: {MAX_WORKERS}")
print(f"Estimated API calls: {estimate_api_calls(len(MODELS))}")


## Validate configuration and connect to OpenRouter

Checks your API key and all prompt templates, then creates the OpenRouter client and a run ID.


In [ ]:
n = len(MODELS)
user_prompt = PRIMARY_PROMPT.strip()
critique_template = CRITIQUE_TEMPLATE.strip()
refinement_template = REFINEMENT_TEMPLATE.strip()
merge_template = MERGE_TEMPLATE.strip()
merge_version_block_template = MERGE_VERSION_BLOCK_TEMPLATE.strip()

if not OPENROUTER_API_KEY:
    raise ValueError(
        "Set OPENROUTER_API_KEY in the configuration cell or export it: "
        "export OPENROUTER_API_KEY='sk-or-...'"
    )
if not user_prompt:
    raise ValueError("PRIMARY_PROMPT is empty.")
if not critique_template:
    raise ValueError("CRITIQUE_TEMPLATE is empty.")
if not refinement_template:
    raise ValueError("REFINEMENT_TEMPLATE is empty.")
if not merge_template:
    raise ValueError("MERGE_TEMPLATE is empty.")
if not merge_version_block_template:
    raise ValueError("MERGE_VERSION_BLOCK_TEMPLATE is empty.")

validate_prompt_templates(
    critique_template,
    refinement_template,
    merge_template,
    merge_version_block_template,
)

client = get_client(OPENROUTER_API_KEY)
run_id = str(uuid.uuid4())[:8]

print(f"Run ID: {run_id}")
print(f"Models ({n}): {MODELS}")
print(f"Merger: {MERGER_MODEL}")
print(f"Temperature: {TEMPERATURE}")
print(f"Estimated API calls: {estimate_api_calls(n)}")
print(f"  primary: {n}")
print(f"  critiques: {n * (n - 1)}")
print(f"  refinements: {n * (n - 1)}")
print(f"  merges: {n}")


## Phase 1 — Primary responses

Each of the **n** author models receives the same `PRIMARY_PROMPT` and produces an independent first answer. Calls run in parallel (up to `MAX_WORKERS` threads) with a tqdm bar.


In [ ]:
primary = run_primary(
    client,
    MODELS,
    user_prompt,
    TEMPERATURE,
    MAX_WORKERS,
)

for i, text in enumerate(primary):
    preview = (text or "")[:300].replace("\n", " ")
    print(f"\n--- LLM {i + 1} ({MODELS[i]}) — {len((text or '').split())} words ---")
    print(preview + ("..." if text and len(text) > 300 else ""))


## Phase 2 — Cross-model critiques

For each pair of distinct models **(i, j)**, model **j** critiques model **i**'s primary response. Result grid: `critiques[i][j]` (diagonal unused).


In [ ]:
critiques = run_critiques(
    client,
    MODELS,
    primary,
    user_prompt,
    critique_template,
    TEMPERATURE,
    MAX_WORKERS,
)

print(f"Critique grid shape: {len(critiques)} × {len(critiques[0])}")


## Phase 3 — Refinements

Each author model **i** revises its primary answer once per peer critique. Result grid: `refined[i][j]` = revision by model **i** after critique from **j**.


In [ ]:
refined = run_refine(
    client,
    MODELS,
    primary,
    critiques,
    user_prompt,
    refinement_template,
    TEMPERATURE,
    MAX_WORKERS,
)

print(f"Refinement grid shape: {len(refined)} × {len(refined[0])}")


## Phase 4 — Merge refinements per author

For each author **i**, the **n−1** refined versions are synthesized into one merged answer using `MERGER_MODEL`.


In [ ]:
merged = run_merges(
    client,
    MERGER_MODEL.strip(),
    MODELS,
    refined,
    user_prompt,
    merge_template,
    merge_version_block_template,
    TEMPERATURE,
)

for i, text in enumerate(merged):
    print(f"LLM {i + 1}: merged response — {len((text or '').split())} words")


## Save run bundle

Package all inputs and outputs into JSON (same schema as the Streamlit app) and optionally write to `OUTPUT_DIR/<run_id>/roundtable_run.json`.


In [ ]:
bundle = build_run_bundle(
    run_id,
    MODELS,
    MERGER_MODEL.strip(),
    TEMPERATURE,
    MAX_WORKERS,
    user_prompt,
    critique_template,
    refinement_template,
    merge_template,
    merge_version_block_template,
    primary,
    critiques,
    refined,
    merged,
)

if AUTO_SAVE:
    saved_to = save_run_bundle_to_dir(bundle, OUTPUT_DIR)
    print(f"Saved to {saved_to / 'roundtable_run.json'}")
else:
    print("AUTO_SAVE is False — bundle kept in memory as `bundle`")


## Compare merged vs primary (TF-IDF cosine)

For each author, we compare the **primary** response to the **merged** response using L2-normalized bag-of-words TF-IDF cosine on a **2-document corpus** (just those two strings).

**IDF** (with `smooth_idf=True`, *n* = 2): `idf(t) = ln((1 + n) / (1 + df(t))) + 1`


In [ ]:
for i in range(n):
    p = primary[i] or ""
    m = merged[i] or ""
    sim = tfidf_cosine(p, m)
    display(Markdown(f"### LLM {i + 1}: `{MODELS[i]}`"))
    print(f"TF-IDF cosine(primary, merged): {sim:.4f}")
    print(f"Words — primary: {len(p.split())}, merged: {len(m.split())}")

    bd = tfidf_pair_breakdown(p, m, top_n=12)
    if bd:
        print(f"Vocab size: {bd['vocab_size']} | IDF formula match: {bd['idf_matches_df_formula']}")
        print("Top primary terms:", [r["term"] for r in bd["primary_top"][:8]])
        print("Top merged terms:", [r["term"] for r in bd["merged_top"][:8]])
    print()


## Side-by-side text and intermediate artifacts

Inspect full primary vs merged text. Set `SHOW_INTERMEDIATES = False` in the configuration cell to skip critiques and refinements.


In [ ]:
for i in range(n):
    display(Markdown(f"## Author LLM {i + 1} — `{MODELS[i]}`"))
    display(Markdown("**Primary**"))
    display(Markdown(primary[i] or "_empty_"))
    display(Markdown("**Merged**"))
    display(Markdown(merged[i] or "_empty_"))

    if SHOW_INTERMEDIATES:
        for j in range(n):
            if j == i:
                continue
            display(Markdown(f"**Critique by LLM {j + 1}**"))
            display(Markdown((critiques[i][j] if critiques else "") or "_missing_"))
            display(Markdown(f"**Refined by LLM {i + 1} after that critique**"))
            display(Markdown((refined[i][j] if refined else "") or "_missing_"))
